In [1]:
import requests
from bs4 import BeautifulSoup
from time import sleep

from io import StringIO
import pandas as pd

import re

In [2]:
def get_response(payload):
    url = "http://siops.datasus.gov.br/consconselho.php"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36',
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    payload["cmbUF"] = "-1"
    payload["cmbMunicipio"] = "-1"
    payload["BtAtualizar"] = "Atualizar"

    s = requests.Session()
    s.headers.update(headers)
    s.get(url)

    response = s.post(url, data = payload, timeout = 10)

    if response.status_code != 200:
        raise Exception(f"Request failed with status code {response.status_code}")
    
    return response


In [3]:
def make_soup(response):
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.select_one('html body div#tudo div#container div.informacao table')

    soup = BeautifulSoup(response.content, 'html.parser')
    title = soup.select_one('html body div#tudo div#container div.informacao div.centro.lbltitulo')
    no_data = title is None

    if table is None:
        if no_data:
            raise Exception("No table found in the HTML content")
        else:
            table = """
            <table>
                <tbody>
                    <tr>
                        <td>Município</td>
                        <td>Código Município</td>
                        <td>Prefeito</td>
                        <td>Secretário</td>
                        <td>Contador</td>
                    </tr>
                </tbody>
            </table>
            """

    return table

In [4]:
def format_dataframe(table):
    table_sep = str(table).replace('<br/>', '===')

    data = pd.read_html(StringIO(table_sep), header = 0)[0]

    data = data.rename(
        columns = {
            "Município": "municipio_nome",
            "Código Município": "municipio",
            "Prefeito": "prefeito",
            "Secretário": "secretario",
            "Contador": "contador",
        },
        inplace = False
    )

    cols = ["prefeito", "secretario", "contador"]

    data[cols] = data[cols].map(
        lambda x: re.sub(r"([^=]+)===.*", r"\1", x).upper()
    )

    return data

In [ ]:
requests_all = {
    "2006_anual": {"cmbANO": "2006", "cmbPERIODO": "2"},
    "2006_1sem": {"cmbANO": "2006", "cmbPERIODO": "1"},

    "2007_anual": {"cmbANO": "2007", "cmbPERIODO": "2"},
    "2007_1sem": {"cmbANO": "2007", "cmbPERIODO": "1"},

    "2008_anual": {"cmbANO": "2008", "cmbPERIODO": "2"},
    "2008_1sem": {"cmbANO": "2008", "cmbPERIODO": "1"},

    "2009_anual": {"cmbANO": "2009", "cmbPERIODO": "2"},
    "2009_1sem": {"cmbANO": "2009", "cmbPERIODO": "1"},

    "2010_anual": {"cmbANO": "2010", "cmbPERIODO": "2"},
    "2010_1sem": {"cmbANO": "2010", "cmbPERIODO": "1"},

    "2011_anual": {"cmbANO": "2011", "cmbPERIODO": "2"},
    "2011_1sem": {"cmbANO": "2011", "cmbPERIODO": "1"},

    "2012_anual": {"cmbANO": "2012", "cmbPERIODO": "2"},
    "2012_1sem": {"cmbANO": "2012", "cmbPERIODO": "1"},

    "2013_1bi": {"cmbANO": "2013", "cmbPERIODO": "12"},
    "2013_2bi": {"cmbANO": "2013", "cmbPERIODO": "14"},
    "2013_3bi": {"cmbANO": "2013", "cmbPERIODO": "1"},
    "2013_4bi": {"cmbANO": "2013", "cmbPERIODO": "18"},
    "2013_5bi": {"cmbANO": "2013", "cmbPERIODO": "20"},
    "2013_6bi": {"cmbANO": "2013", "cmbPERIODO": "2"},

    "2014_1bi": {"cmbANO": "2014", "cmbPERIODO": "12"},
    "2014_2bi": {"cmbANO": "2014", "cmbPERIODO": "14"},
    "2014_3bi": {"cmbANO": "2014", "cmbPERIODO": "1"},
    "2014_4bi": {"cmbANO": "2014", "cmbPERIODO": "18"},
    "2014_5bi": {"cmbANO": "2014", "cmbPERIODO": "20"},
    "2014_6bi": {"cmbANO": "2014", "cmbPERIODO": "2"},

    "2015_1bi": {"cmbANO": "2015", "cmbPERIODO": "12"},
    "2015_2bi": {"cmbANO": "2015", "cmbPERIODO": "14"},
    "2015_3bi": {"cmbANO": "2015", "cmbPERIODO": "1"},
    "2015_4bi": {"cmbANO": "2015", "cmbPERIODO": "18"},
    "2015_5bi": {"cmbANO": "2015", "cmbPERIODO": "20"},
    "2015_6bi": {"cmbANO": "2015", "cmbPERIODO": "2"},

    "2016_1bi": {"cmbANO": "2016", "cmbPERIODO": "12"},
    "2016_2bi": {"cmbANO": "2016", "cmbPERIODO": "14"},
    "2016_3bi": {"cmbANO": "2016", "cmbPERIODO": "1"},
    "2016_4bi": {"cmbANO": "2016", "cmbPERIODO": "18"},
    "2016_5bi": {"cmbANO": "2016", "cmbPERIODO": "20"},
    "2016_6bi": {"cmbANO": "2016", "cmbPERIODO": "2"},

    "2017_1bi": {"cmbANO": "2017", "cmbPERIODO": "12"},
    "2017_2bi": {"cmbANO": "2017", "cmbPERIODO": "14"},
    "2017_3bi": {"cmbANO": "2017", "cmbPERIODO": "1"},
    "2017_4bi": {"cmbANO": "2017", "cmbPERIODO": "18"},
    "2017_5bi": {"cmbANO": "2017", "cmbPERIODO": "20"},
    "2017_6bi": {"cmbANO": "2017", "cmbPERIODO": "2"},
}

# Initialize loop, considering all keys
still_to_process = list(requests_all.keys())
data_list = dict()

In [ ]:
requests_to_process = {key: requests_all[key] for key in still_to_process}

for key, payload in requests_to_process.items():
    print(f"Processing {key}...")

    try:
        response = get_response(payload)
    except:
        data_list[key] = None
        sleep(2)
        continue

    table = make_soup(response)
    data = format_dataframe(table)

    key_split = key.split("_")
    data["ano"] = key_split[0]
    data["periodo"] = key_split[1]

    data_list[key] = data

    sleep(2) if data.empty else sleep(0.5)

In [ ]:
response_fails = [key for key, df in data_list.items() if df is None]
print("Requests that failed:", response_fails)

x = data_all["ano"] + "_" + data_all["periodo"]
missing_keys = list(
    set(requests_all.keys()) - set(x.drop_duplicates()) - set(response_fails)
)
print("Tables with no data:", missing_keys)

# Update the list of keys to process, then re-run the loop above
still_to_process = response_fails

In [ ]:
# After no more response_fails, combine all data and save to CSV
data_all = pd.concat(data_list)
data_all.to_csv("data_all.csv", index = False)